In [2]:
# --- STEP 1: 5-FOLD STRATIFIED CV GAIN SCREENING (FULL DATASET) ---

import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

start_time = time.time()
print("🚀 [STEP 1] Bắt đầu đánh giá Feature Importance (Gain) bằng 5-Fold CV...")

# 1. Nạp Master Dataset V2
master_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v2.parquet'
print(f"⏳ Đang nạp toàn bộ Master Dataset từ: {master_path}")
df_master = pd.read_parquet(master_path)

drop_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_master.columns if c not in drop_cols]

X = df_master[feature_cols].copy()
y = df_master['TARGET']

# Ép kiểu category cho toàn bộ cột chuỗi (string/object)
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
if cat_cols:
    print(f"💡 Đã chuyển đổi {len(cat_cols)} cột văn bản sang kiểu 'category'.")
    for col in cat_cols:
        X[col] = X[col].astype('category')

total_rows, total_cols = X.shape
print(f"📊 Dữ liệu đầu vào: {total_rows:,} dòng | {total_cols} thuộc tính")
print("-" * 80)

# 2. Khởi tạo Stratified 5-Fold Cross-Validation
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

feature_importances_gain = np.zeros(total_cols)
trained_models = []  # Lưu lại 5 models để dùng tiếp cho Step 2 nếu cần

# 3. Vòng lặp 5-Fold CV
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    fold_start = time.time()
    print(f"⏳ [Fold {fold}/{N_SPLITS}] Đang huấn luyện LightGBM...")

    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    feature_importances_gain += model.feature_importances_ / N_SPLITS
    trained_models.append((model, val_idx))

    print(f"   ✓ Fold {fold} hoàn tất trong: {time.time() - fold_start:.2f} giây")

# 4. Tổng hợp bảng xếp hạng Gain
df_gain_ranking = pd.DataFrame({
    'feature': feature_cols,
    'mean_gain': feature_importances_gain
}).sort_values('mean_gain', ascending=False).reset_index(drop=True)

zero_imp_df = df_gain_ranking[df_gain_ranking['mean_gain'] == 0]
useful_imp_df = df_gain_ranking[df_gain_ranking['mean_gain'] > 0]
zero_importance_cols = zero_imp_df['feature'].tolist()

# Xuất file kết quả Step 1
output_step1_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/feature_gain_ranking_5fold.csv'
os.makedirs(os.path.dirname(output_step1_path), exist_ok=True)
df_gain_ranking.to_csv(output_step1_path, index=False)

print("=" * 80)
print(f"🎉 HOÀN TẤT STEP 1 CẢ 5-FOLD CV TRONG: {time.time() - start_time:.2f} GIÂY")
print(f"💾 Kết quả Gain đã lưu tại: {output_step1_path}")
print(f"✅ Số lượng cột CÓ ĐÓNG GÓP (Mean Gain > 0) : {len(useful_imp_df):,} cột")
print(f"❌ Số lượng cột NHIỄU / THỪA (Mean Gain == 0): {len(zero_importance_cols):,} cột")
print("=" * 80)

# In danh sách code Python để copy xóa
print("\n" + "=" * 80)
print(f"📌 DANH SÁCH {len(zero_importance_cols)} CỘT BỊ 0 ĐIỂM GAIN (COPY ĐỂ DROP):")
print("=" * 80)
print("zero_importance_cols = [")
for col in zero_importance_cols:
    print(f"    '{col}',")
print("]")

🚀 [STEP 1] Bắt đầu đánh giá Feature Importance (Gain) bằng 5-Fold CV...
⏳ Đang nạp toàn bộ Master Dataset từ: /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v2.parquet
💡 Đã chuyển đổi 6 cột văn bản sang kiểu 'category'.
📊 Dữ liệu đầu vào: 307,511 dòng | 1030 thuộc tính
--------------------------------------------------------------------------------
⏳ [Fold 1/5] Đang huấn luyện LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.163600 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 162742
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 1029
[LightGBM] [Warning] Found whitespace in feature_names, replace with underl

In [4]:
# --- STEP 2: 5-FOLD STRATIFIED CV TREESHAP EVALUATION (FIXED SHAPE ERROR) ---

import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import shap

start_time = time.time()
print("🚀 [STEP 2] Bắt đầu tính toán TreeSHAP Values trên tập cột có Gain > 0...")

# 1. Lọc dữ liệu chỉ giữ lại các cột có Gain > 0 từ Step 1
valid_features = useful_imp_df['feature'].tolist()
X_filtered = X[valid_features].copy()

total_rows, total_cols = X_filtered.shape
print(f"📊 Tập thuộc tính đưa vào SHAP: {total_cols} cột (Đã cắt bỏ {len(zero_importance_cols)} cột nhiễu)")
print("-" * 80)

# 2. Khởi tạo 5-Fold CV mới cho tập thuộc tính đã rút gọn
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

shap_importances_mean = np.zeros(total_cols)

# 3. Vòng lặp huấn luyện & tính SHAP trực tiếp trên tập thuộc tính sạch
for fold, (train_idx, val_idx) in enumerate(skf.split(X_filtered, y), 1):
    fold_start = time.time()
    print(f"⏳ [Fold {fold}/{N_SPLITS}] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...")

    X_train, y_train = X_filtered.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_filtered.iloc[val_idx], y.iloc[val_idx]

    # Model LightGBM huấn luyện riêng trên tập cột đã lọc sạch
    model_clean = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model_clean.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # Lấy mẫu 3,000 khách hàng/fold để tính SHAP an toàn bộ nhớ RAM
    X_val_sample = X_val.sample(n=min(3000, len(X_val)), random_state=42)

    explainer = shap.TreeExplainer(model_clean)
    shap_vals = explainer.shap_values(X_val_sample)

    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    shap_importances_mean += np.abs(shap_vals).mean(axis=0) / N_SPLITS
    print(f"   ✓ Fold {fold} SHAP hoàn tất trong: {time.time() - fold_start:.2f} giây")

# 4. Gộp điểm Mean Gain (Step 1) và Mean Abs SHAP (Step 2)
df_final_ranking = useful_imp_df.copy()
df_final_ranking['mean_abs_shap'] = shap_importances_mean
df_final_ranking = df_final_ranking.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# 5. Xuất file xếp hạng tổng hợp
output_step2_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/feature_ranking_5fold_shap.csv'
os.makedirs(os.path.dirname(output_step2_path), exist_ok=True)
df_final_ranking.to_csv(output_step2_path, index=False)

print("=" * 80)
print(f"🎉 HOÀN TẤT STEP 2 SHAP RANKING TRONG: {time.time() - start_time:.2f} GIÂY")
print(f"💾 File xếp hạng đầy đủ (Gain + SHAP) đã lưu tại: {output_step2_path}")
print(f"📊 Tổng số thuộc tính được chấm điểm SHAP: {len(df_final_ranking):,} cột")
print("=" * 80)

print("\n📋 TOP 50 CỘT CÓ ĐIỂM SHAP CAO NHẤT TỪ 5-FOLD CV:")
display(df_final_ranking.head(50))

🚀 [STEP 2] Bắt đầu tính toán TreeSHAP Values trên tập cột có Gain > 0...
📊 Tập thuộc tính đưa vào SHAP: 876 cột (Đã cắt bỏ 154 cột nhiễu)
--------------------------------------------------------------------------------
⏳ [Fold 1/5] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.192369 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157525
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.43248

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✓ Fold 1 SHAP hoàn tất trong: 36.21 giây
⏳ [Fold 2/5] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.181985 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157499
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✓ Fold 2 SHAP hoàn tất trong: 40.02 giây
⏳ [Fold 3/5] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.269560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157467
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


...

   ✓ Fold 3 SHAP hoàn tất trong: 33.35 giây
⏳ [Fold 4/5] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.158188 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157546
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


   ✓ Fold 4 SHAP hoàn tất trong: 41.08 giây
⏳ [Fold 5/5] Huấn luyện mô hình tinh gọn & Tính TreeSHAP...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.137528 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157500
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
   ✓ Fold 5 SHAP hoàn tất trong: 35.64 giây
🎉 HOÀN TẤT STEP 2 SHAP RANKING TRONG: 189.49 GIÂY
💾 File xếp hạng đầy đủ (Gain + SHAP) đã lưu tại: /Users/nguyenminhtri/FinalYear

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_gain,mean_abs_shap
0,EXT_SOURCE_MEAN,112244.736697,0.491528
1,CODE_GENDER,2481.160104,0.085117
2,BUREAU_DEBT_RATIO_MAX,6293.018353,0.074687
3,PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,3010.383031,0.071406
4,CREDIT_TO_ANNUITY,7301.445893,0.068971
5,AMT_ANNUITY,3217.853413,0.068399
6,NAME_EDUCATION_TYPE,3104.740696,0.066998
7,INS_IS_PAYMENT_DELAYED_MEAN,4428.005551,0.064258
8,ORGANIZATION_TYPE,10194.455316,0.063804
9,EXT_SOURCE_1,4422.943121,0.061462


In [5]:
# --- INTEGRATED CELL: 5-FOLD CV SCREENING, SHAP RANKING & MASTER V3 EXPORT ---

import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import shap

start_time = time.time()
print("🚀 Bắt đầu quá trình 5-Fold CV Screening, SHAP Ranking và Xuất Dataset sạch...")

# 1. Nạp Master Dataset V2
v2_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v2.parquet'
v3_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v3.parquet'

print(f"⏳ Đang nạp Master Dataset V2 từ: {v2_path}")
df_master = pd.read_parquet(v2_path)

drop_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_master.columns if c not in drop_cols]

X = df_master[feature_cols].copy()
y = df_master['TARGET']

# Ép kiểu category cho toàn bộ cột chuỗi (string/object) để chống lỗi LightGBM
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
if cat_cols:
    for col in cat_cols:
        X[col] = X[col].astype('category')

total_rows, total_cols = X.shape
print(f"📊 Dữ liệu đầu vào: {total_rows:,} dòng | {total_cols} thuộc tính")
print("-" * 80)

# 2. STEP 1: Khởi tạo & Huấn luyện 5-Fold CV để tính Mean Gain
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

feature_importances_gain = np.zeros(total_cols)

print("⏳ [STEP 1] Đang chạy 5-Fold CV để sàng lọc Feature Importance (Gain)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    feature_importances_gain += model.feature_importances_ / N_SPLITS

# Bảng xếp hạng Gain
df_gain = pd.DataFrame({
    'feature': feature_cols,
    'mean_gain': feature_importances_gain
})

useful_features = df_gain[df_gain['mean_gain'] > 0]['feature'].tolist()
zero_importance_cols = df_gain[df_gain['mean_gain'] == 0]['feature'].tolist()

print(f"✅ Số cột CÓ ĐÓNG GÓP (Gain > 0) : {len(useful_features):,} cột")
print(f"❌ Số cột NHIỄU / THỪA (Gain == 0): {len(zero_importance_cols):,} cột")
print("-" * 80)

# 3. STEP 2: Tính TreeSHAP trên tập cột sạch (Gain > 0)
print(f"⏳ [STEP 2] Đang tính TreeSHAP Values trên {len(useful_features)} cột có Gain > 0...")
X_filtered = X[useful_features].copy()
shap_importances_mean = np.zeros(len(useful_features))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_filtered, y), 1):
    X_train, y_train = X_filtered.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_filtered.iloc[val_idx], y.iloc[val_idx]

    model_clean = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model_clean.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    X_val_sample = X_val.sample(n=min(3000, len(X_val)), random_state=42)
    explainer = shap.TreeExplainer(model_clean)
    shap_vals = explainer.shap_values(X_val_sample)

    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    shap_importances_mean += np.abs(shap_vals).mean(axis=0) / N_SPLITS

# Xuất file CSV điểm số xếp hạng SHAP
df_final_ranking = df_gain[df_gain['mean_gain'] > 0].copy().reset_index(drop=True)
df_final_ranking['mean_abs_shap'] = shap_importances_mean
df_final_ranking = df_final_ranking.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

csv_ranking_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/feature_ranking_5fold_shap.csv'
os.makedirs(os.path.dirname(csv_ranking_path), exist_ok=True)
df_final_ranking.to_csv(csv_ranking_path, index=False)

# 4. STEP 3: TỰ ĐỘNG DROP 154 CỘT NHIỄU & XUẤT FILE MASTER V3 DUY NHẤT
print("-" * 80)
print(f"✂️ [STEP 3] Đang tiến hành loại bỏ {len(zero_importance_cols)} cột nhiễu khỏi Master Dataset...")

df_master_clean = df_master.drop(columns=zero_importance_cols, errors='ignore')

os.makedirs(os.path.dirname(v3_path), exist_ok=True)
print(f"⏳ Đang lưu Master Dataset V3 sạch ra đĩa tại: {v3_path}")
df_master_clean.to_parquet(v3_path, compression='snappy', index=False)

exec_time = time.time() - start_time
print("=" * 80)
print(f"🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH TRONG: {exec_time:.2f} GIÂY")
print(f"💾 File xếp hạng điểm số SHAP đã lưu : {csv_ranking_path}")
print(f"💾 File Master Dataset V3 sạch đã lưu : {v3_path}")
print(
    f"📊 Kích thước bảng Master V3 mới       : {df_master_clean.shape[0]:,} dòng | {df_master_clean.shape[1]} thuộc tính")
print(f"📉 Đã cắt giảm                         : {total_cols + len(drop_cols) - df_master_clean.shape[1]} cột nhiễu")
print("=" * 80)

# Hiển thị Top 50 cột có điểm SHAP cao nhất
display(df_final_ranking.head(50))

🚀 Bắt đầu quá trình 5-Fold CV Screening, SHAP Ranking và Xuất Dataset sạch...
⏳ Đang nạp Master Dataset V2 từ: /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v2.parquet
📊 Dữ liệu đầu vào: 307,511 dòng | 1030 thuộc tính
--------------------------------------------------------------------------------
⏳ [STEP 1] Đang chạy 5-Fold CV để sàng lọc Feature Importance (Gain)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.190098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 162742
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 1029
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Inf

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.141286 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157499
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.132237 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157467
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


...

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.154314 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157500
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 876
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


--------------------------------------------------------------------------------
✂️ [STEP 3] Đang tiến hành loại bỏ 154 cột nhiễu khỏi Master Dataset...
⏳ Đang lưu Master Dataset V3 sạch ra đĩa tại: /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v3.parquet
🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH TRONG: 383.77 GIÂY
💾 File xếp hạng điểm số SHAP đã lưu : /Users/nguyenminhtri/FinalYearPro/data/processed/feature_ranking_5fold_shap.csv
💾 File Master Dataset V3 sạch đã lưu : /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v3.parquet
📊 Kích thước bảng Master V3 mới       : 307,511 dòng | 878 thuộc tính
📉 Đã cắt giảm                         : 154 cột nhiễu


,feature,mean_gain,mean_abs_shap
0,EXT_SOURCE_MEAN,112244.736697,0.491530
1,CODE_GENDER,2481.160104,0.085116
2,BUREAU_DEBT_RATIO_MAX,6293.018353,0.074687
3,PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,3010.383031,0.071416
4,CREDIT_TO_ANNUITY,7301.445893,0.068975
5,AMT_ANNUITY,3217.853413,0.068345
6,NAME_EDUCATION_TYPE,3104.740696,0.066997
7,INS_IS_PAYMENT_DELAYED_MEAN,4428.005551,0.064255
8,ORGANIZATION_TYPE,10194.455316,0.063804
9,EXT_SOURCE_1,4422.943121,0.061462


In [7]:
# --- INTEGRATED CELL: 5-FOLD CV SCREENING, SHAP RANKING & MASTER V4 CLEAN EXPORT ---

import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import shap

start_time = time.time()
print("🚀 Bắt đầu quá trình 5-Fold CV Screening, SHAP Ranking và Xuất Master V4 Clean...")

# 1. Đường dẫn file nạp vào và xuất ra
input_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v4.parquet'
output_clean_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v4_clean.parquet'

print(f"⏳ Đang nạp Master Dataset V4 từ: {input_path}")
df_master = pd.read_parquet(input_path)

drop_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_master.columns if c not in drop_cols]

X = df_master[feature_cols].copy()
y = df_master['TARGET']

# Ép kiểu category cho toàn bộ cột chuỗi (string/object) để LightGBM xử lý tối ưu
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
if cat_cols:
    for col in cat_cols:
        X[col] = X[col].astype('category')

total_rows, total_cols = X.shape
print(f"📊 Dữ liệu đầu vào: {total_rows:,} dòng | {total_cols} thuộc tính")
print("-" * 80)

# 2. STEP 1: Khởi tạo & Huấn luyện 5-Fold CV để tính Mean Gain
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

feature_importances_gain = np.zeros(total_cols)

print("⏳ [STEP 1] Đang chạy 5-Fold CV để sàng lọc Feature Importance (Gain)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    feature_importances_gain += model.feature_importances_ / N_SPLITS

# Bảng xếp hạng Gain
df_gain = pd.DataFrame({
    'feature': feature_cols,
    'mean_gain': feature_importances_gain
})

useful_features = df_gain[df_gain['mean_gain'] > 0]['feature'].tolist()
zero_importance_cols = df_gain[df_gain['mean_gain'] == 0]['feature'].tolist()

print(f"✅ Số cột CÓ ĐÓNG GÓP (Gain > 0) : {len(useful_features):,} cột")
print(f"❌ Số cột NHIỄU / THỪA (Gain == 0): {len(zero_importance_cols):,} cột")
print("-" * 80)

# 3. STEP 2: Tính TreeSHAP trên tập cột sạch (Gain > 0)
print(f"⏳ [STEP 2] Đang tính TreeSHAP Values trên {len(useful_features)} cột có Gain > 0...")
X_filtered = X[useful_features].copy()
shap_importances_mean = np.zeros(len(useful_features))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_filtered, y), 1):
    X_train, y_train = X_filtered.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X_filtered.iloc[val_idx], y.iloc[val_idx]

    model_clean = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42 + fold,
        n_jobs=-1,
        importance_type='gain'
    )

    model_clean.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    X_val_sample = X_val.sample(n=min(3000, len(X_val)), random_state=42)
    explainer = shap.TreeExplainer(model_clean)
    shap_vals = explainer.shap_values(X_val_sample)

    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    shap_importances_mean += np.abs(shap_vals).mean(axis=0) / N_SPLITS

# Tạo bảng hiển thị SHAP (không lưu ra file CSV)
df_final_ranking = df_gain[df_gain['mean_gain'] > 0].copy().reset_index(drop=True)
df_final_ranking['mean_abs_shap'] = shap_importances_mean
df_final_ranking = df_final_ranking.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# 4. STEP 3: TỰ ĐỘNG DROP CÁC CỘT NHIỄU & XUẤT FILE MASTER V4 SẠCH
print("-" * 80)
print(f"✂️ [STEP 3] Đang tiến hành loại bỏ {len(zero_importance_cols)} cột nhiễu khỏi Master Dataset...")

df_master_clean = df_master.drop(columns=zero_importance_cols, errors='ignore')

os.makedirs(os.path.dirname(output_clean_path), exist_ok=True)
print(f"⏳ Đang lưu Master Dataset V4 sạch ra đĩa tại: {output_clean_path}")
df_master_clean.to_parquet(output_clean_path, compression='snappy', index=False)

exec_time = time.time() - start_time
print("=" * 80)
print(f"🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH TRONG: {exec_time:.2f} GIÂY")
print(f"💾 File Master Dataset V4 sạch đã lưu : {output_clean_path}")
print(f"📊 Kích thước bảng Master V4 Clean    : {df_master_clean.shape[0]:,} dòng | {df_master_clean.shape[1]} thuộc tính")
print(f"📉 Đã cắt giảm                        : {len(zero_importance_cols)} cột nhiễu")
print("=" * 80)

# Hiển thị Top 50 cột có điểm SHAP cao nhất trên notebook
display(df_final_ranking.head(50))

🚀 Bắt đầu quá trình 5-Fold CV Screening, SHAP Ranking và Xuất Master V4 Clean...
⏳ Đang nạp Master Dataset V4 từ: /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v4.parquet
📊 Dữ liệu đầu vào: 307,511 dòng | 1041 thuộc tính
--------------------------------------------------------------------------------
⏳ [STEP 1] Đang chạy 5-Fold CV để sàng lọc Feature Importance (Gain)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.173885 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 164633
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 1040
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.168090 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 160232
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 902
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.168371 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 160195
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 902
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


...

/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.141750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 160249
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 902
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


/Users/nguyenminhtri/FinalYearPro/venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


--------------------------------------------------------------------------------
✂️ [STEP 3] Đang tiến hành loại bỏ 139 cột nhiễu khỏi Master Dataset...
⏳ Đang lưu Master Dataset V4 sạch ra đĩa tại: /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v4_clean.parquet
🎉 HOÀN TẤT TOÀN BỘ QUY TRÌNH TRONG: 416.69 GIÂY
💾 File Master Dataset V4 sạch đã lưu : /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v4_clean.parquet
📊 Kích thước bảng Master V4 Clean    : 307,511 dòng | 904 thuộc tính
📉 Đã cắt giảm                        : 139 cột nhiễu


,feature,mean_gain,mean_abs_shap
0,EXT_SOURCE_MEAN,112333.048635,0.489782
1,CODE_GENDER,2535.248502,0.085438
2,BUREAU_DEBT_RATIO_MAX,6327.265509,0.076007
3,PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,3127.956589,0.074173
4,CREDIT_TO_ANNUITY,7226.731169,0.070061
5,AMT_ANNUITY,3312.176153,0.068535
6,NAME_EDUCATION_TYPE,3128.533528,0.067651
7,ORGANIZATION_TYPE,10651.244981,0.065253
8,INS_IS_PAYMENT_DELAYED_MEAN,4497.289970,0.065197
9,EXT_SOURCE_1,4473.696437,0.061960
